# India Road Safety Dashboard
### Data Source: MoRTH Annual Reports (2015–2023) & NCRB Data
**Author:** Albert Lohe | M.Tech Safety Engineering & Analytics, IIT Kharagpur  
**Focus:** Motorcycle accident trends,policy impact analysis, and national safety KPIs

---
## Project Structure
| Section | Description |
|---------|-------------|
| 1. Setup | Imports, configuration |
| 2. Data Loading | Load & inspect all datasets |
| 3. Data Cleaning | Preprocessing, feature engineering |
| 4. EDA | Exploratory analysis with key statistics |
| 5. National Trends | Year-wise accident, fatality, injury trends |
| 6. Motorcycle Analysis | Two-wheeler specific deep dive |
| 7. ITS Analysis | Interrupted Time Series – AHO mandate (April 2017) |
| 8. State-wise Analysis | Geographic breakdown |
| 9. Cause Analysis | Accident cause distribution |
| 10. KPI Dashboard | Summary metrics & insights |


## 1. Setup & Imports

In [1]:
# In Jupyter, use the shell bang prefix and valid package names
!pip install plotly statsmodels
!pip install plotly statsmodels nbformat>=4.2.0
print("\n📊 Chart rendered! Key observation: Fatality rate peaks in 2023 despite fewer accidents than 2015.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



📊 Chart rendered! Key observation: Fatality rate peaks in 2023 despite fewer accidents than 2015.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Core Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'vscode'

# Stats & Modeling
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from scipy import stats

# Display
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Config
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("✅ All libraries loaded successfully!")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")


✅ All libraries loaded successfully!
Pandas: 3.0.0 | NumPy: 2.4.1


## 2. Data Loading
**Data Sources:**
- `morth_accidents.csv` — National annual accident statistics (MoRTH Annual Reports)
- `statewise_accidents.csv` — State-wise breakdown
- `monthly_motorcycle.csv` — Monthly motorcycle data for ITS analysis
- `cause_wise.csv` — Accident cause distribution


In [3]:
# Load datasets
df_national  = pd.read_csv('data/morth_accidents.csv')
df_state     = pd.read_csv('data/statewise_accidents.csv')
df_monthly   = pd.read_csv('data/monthly_motorcycle.csv')
df_cause     = pd.read_csv('data/cause_wise.csv')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
for name, df in [("National Annual", df_national),
                 ("State-wise",      df_state),
                 ("Monthly Motorcycle", df_monthly),
                 ("Cause-wise",      df_cause)]:
    print(f"\n📂 {name}: {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"   Columns: {list(df.columns)}")


DATASET OVERVIEW

📂 National Annual: 9 rows × 15 cols
   Columns: ['Year', 'Total_Accidents', 'Total_Fatalities', 'Total_Injuries', 'Motorcycle_Accidents', 'Motorcycle_Fatalities', 'Motorcycle_Injuries', 'NH_Accidents', 'SH_Accidents', 'Other_Road_Accidents', 'Pedestrian_Deaths', 'Cyclist_Deaths', 'Truck_Involved', 'Car_Involved', 'Two_Wheeler_Involved']

📂 State-wise: 60 rows × 6 cols
   Columns: ['State', 'Year', 'Accidents', 'Fatalities', 'Injuries', 'Fatality_Rate']

📂 Monthly Motorcycle: 108 rows × 7 cols
   Columns: ['Year', 'Month', 'Month_Num', 'Motorcycle_Accidents', 'Motorcycle_Fatalities', 'Post_AHO', 'Time_Index']

📂 Cause-wise: 12 rows × 7 cols
   Columns: ['Cause', 'Accidents_2019', 'Accidents_2021', 'Accidents_2023', 'Fatalities_2019', 'Fatalities_2021', 'Fatalities_2023']


### 2.1 Quick Inspection

In [4]:
print("── National Data (Head) ──────────────────────────")
display(df_national.head())

print("\n── Statistical Summary ──────────────────────────")
display(df_national.describe())

print("\n── Missing Values Check ─────────────────────────")
for name, df in [("National", df_national), ("State", df_state),
                 ("Monthly", df_monthly), ("Cause", df_cause)]:
    nulls = df.isnull().sum().sum()
    print(f"  {name:20s}: {nulls} missing values")


── National Data (Head) ──────────────────────────


,Year,Total_Accidents,Total_Fatalities,Total_Injuries,Motorcycle_Accidents,Motorcycle_Fatalities,Motorcycle_Injuries,NH_Accidents,SH_Accidents,Other_Road_Accidents,Pedestrian_Deaths,Cyclist_Deaths,Truck_Involved,Car_Involved,Two_Wheeler_Involved
0,2015,501423,146133,500279,153845,50218,148930,155293,95270,250860,15693,4813,85242,95271,153845
1,2016,480652,150785,494624,157320,52384,152100,148521,91324,240807,15902,4995,82451,91490,157320
2,2017,464910,147913,470975,153230,50945,148220,145842,88791,230277,15431,4712,80234,88447,153230
3,2018,467044,151417,469418,154820,52590,149810,148931,90245,227868,15823,4831,81234,90112,154820
4,2019,449002,151113,451361,149720,51890,145920,141732,87342,219928,15672,4721,79823,87542,149720



── Statistical Summary ──────────────────────────


,Year,Total_Accidents,Total_Fatalities,Total_Injuries,Motorcycle_Accidents,Motorcycle_Fatalities,Motorcycle_Injuries,NH_Accidents,SH_Accidents,Other_Road_Accidents,Pedestrian_Deaths,Cyclist_Deaths,Truck_Involved,Car_Involved,Two_Wheeler_Involved
count,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00,9.00
mean,"2,019.00","454,262.22","152,648.11","447,326.22","149,648.33","52,561.44","145,030.00","144,305.89","88,170.44","221,785.89","15,853.11","4,754.44","79,145.00","87,699.89","149,648.33"
std,2.74,"41,477.12","11,985.40","50,169.60","11,719.04","4,399.18","11,207.74","12,694.97","7,334.79","22,292.69","1,235.23",371.52,"6,394.25","7,362.11","11,719.04"
min,"2,015.00","366,138.00","131,714.00","348,279.00","124,190.00","44,840.00","120,230.00","116,731.00","72,183.00","177,224.00","13,421.00","3,923.00","65,234.00","71,849.00","124,190.00"
25%,"2,017.00","449,002.00","147,913.00","443,366.00","149,720.00","50,945.00","145,920.00","141,732.00","87,342.00","219,757.00","15,632.00","4,712.00","79,823.00","87,542.00","149,720.00"
50%,"2,019.00","464,910.00","151,113.00","463,186.00","153,260.00","52,140.00","148,730.00","148,521.00","90,245.00","227,868.00","15,693.00","4,813.00","81,234.00","90,112.00","153,260.00"
75%,"2,021.00","480,652.00","153,972.00","470,975.00","154,820.00","52,590.00","149,810.00","150,321.00","91,324.00","231,485.00","15,902.00","4,995.00","82,451.00","91,490.00","154,820.00"
max,"2,023.00","501,423.00","172,295.00","500,279.00","162,890.00","60,234.00","157,420.00","158,341.00","95,621.00","250,860.00","17,891.00","5,231.00","85,242.00","95,271.00","162,890.00"



── Missing Values Check ─────────────────────────
  National            : 0 missing values
  State               : 0 missing values
  Monthly             : 0 missing values
  Cause               : 0 missing values


## 3. Data Cleaning & Feature Engineering

In [5]:
# ── Feature Engineering on National Data ──────────────────────
df_national['Fatality_Rate']         = (df_national['Total_Fatalities'] / df_national['Total_Accidents'] * 100).round(2)
df_national['Moto_Accident_Share']   = (df_national['Motorcycle_Accidents'] / df_national['Total_Accidents'] * 100).round(2)
df_national['Moto_Fatality_Share']   = (df_national['Motorcycle_Fatalities'] / df_national['Total_Fatalities'] * 100).round(2)
df_national['YoY_Accident_Change']   = df_national['Total_Accidents'].pct_change().round(4) * 100
df_national['YoY_Fatality_Change']   = df_national['Total_Fatalities'].pct_change().round(4) * 100

# ── Monthly Data: Date column ───────────────────────────────────
df_monthly['Date'] = pd.to_datetime(
    df_monthly['Year'].astype(str) + '-' + df_monthly['Month_Num'].astype(str).str.zfill(2) + '-01'
)
df_monthly['Fatality_Rate'] = (df_monthly['Motorcycle_Fatalities'] / df_monthly['Motorcycle_Accidents'] * 100).round(2)

# ── Post-AHO flag label ─────────────────────────────────────────
df_monthly['Period'] = df_monthly['Post_AHO'].map({0: 'Pre-AHO (Before Apr 2017)', 1: 'Post-AHO (Apr 2017+)'})

print("✅ Feature engineering complete!")
print("\n── Engineered Features Added ────────────────────")
print("  df_national: Fatality_Rate, Moto_Accident_Share, Moto_Fatality_Share, YoY changes")
print("  df_monthly:  Date column, Fatality_Rate, Period label")

display(df_national[['Year','Total_Accidents','Total_Fatalities','Fatality_Rate',
                      'Moto_Accident_Share','Moto_Fatality_Share','YoY_Accident_Change']].round(2))


✅ Feature engineering complete!

── Engineered Features Added ────────────────────
  df_national: Fatality_Rate, Moto_Accident_Share, Moto_Fatality_Share, YoY changes
  df_monthly:  Date column, Fatality_Rate, Period label


,Year,Total_Accidents,Total_Fatalities,Fatality_Rate,Moto_Accident_Share,Moto_Fatality_Share,YoY_Accident_Change
0,2015,501423,146133,29.14,30.68,34.36,NaN
1,2016,480652,150785,31.37,32.73,34.74,-4.14
2,2017,464910,147913,31.82,32.96,34.44,-3.28
3,2018,467044,151417,32.42,33.15,34.73,0.46
4,2019,449002,151113,33.66,33.35,34.34,-3.86
5,2020,366138,131714,35.97,33.92,34.04,-18.46
6,2021,412432,153972,37.33,33.35,33.86,12.64
7,2022,461312,168491,36.52,33.22,34.31,11.85
8,2023,485447,172295,35.49,33.55,34.96,5.23


## 4. Exploratory Data Analysis

In [6]:
print("=" * 60)
print("KEY STATISTICS (2015–2023)")
print("=" * 60)

total_deaths  = df_national['Total_Fatalities'].sum()
avg_deaths    = df_national['Total_Fatalities'].mean()
peak_year     = df_national.loc[df_national['Total_Fatalities'].idxmax(), 'Year']
moto_avg_share = df_national['Moto_Fatality_Share'].mean()

pre_aho  = df_national[df_national['Year'] < 2017]['Total_Fatalities'].mean()
post_aho = df_national[df_national['Year'] >= 2017]['Total_Fatalities'].mean()

print(f"\n  Total road deaths (2015-23) : {total_deaths:,}")
print(f"  Average annual deaths       : {avg_deaths:,.0f}")
print(f"  Peak fatality year          : {peak_year}")
print(f"  Avg motorcycle fatality %   : {moto_avg_share:.1f}%")
print(f"  Avg deaths Pre-AHO (15-16)  : {pre_aho:,.0f}")
print(f"  Avg deaths Post-AHO (17-23) : {post_aho:,.0f}")
print(f"  Change post-AHO mandate     : {((post_aho-pre_aho)/pre_aho)*100:+.1f}%")

print("\n── Correlation Matrix (National) ─────────────────")
corr_cols = ['Total_Accidents','Total_Fatalities','Motorcycle_Accidents',
             'Motorcycle_Fatalities','Fatality_Rate']
display(df_national[corr_cols].corr().round(3))


KEY STATISTICS (2015–2023)

  Total road deaths (2015-23) : 1,373,833
  Average annual deaths       : 152,648
  Peak fatality year          : 2023
  Avg motorcycle fatality %   : 34.4%
  Avg deaths Pre-AHO (15-16)  : 148,459
  Avg deaths Post-AHO (17-23) : 153,845
  Change post-AHO mandate     : +3.6%

── Correlation Matrix (National) ─────────────────


,Total_Accidents,Total_Fatalities,Motorcycle_Accidents,Motorcycle_Fatalities,Fatality_Rate
Total_Accidents,1.00,0.53,0.95,0.58,-0.64
Total_Fatalities,0.53,1.00,0.69,0.99,0.31
Motorcycle_Accidents,0.95,0.69,1.00,0.74,-0.46
Motorcycle_Fatalities,0.58,0.99,0.74,1.00,0.24
Fatality_Rate,-0.64,0.31,-0.46,0.24,1.00


## 5. National Road Safety Trends (2015–2023)

In [7]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Accidents & Fatalities (2015–2023)',
        'Year-on-Year Change (%)',
        'Fatality Rate Trend',
        'Road Type — Accident Distribution (2023)'
    ),
    specs=[[{"secondary_y": True}, {}],
           [{}, {"type": "pie"}]]
)

years = df_national['Year']

# ── Plot 1: Accidents + Fatalities ─────────────────────────────
fig.add_trace(go.Bar(x=years, y=df_national['Total_Accidents'],
                      name='Total Accidents', marker_color='steelblue', opacity=0.7),
              row=1, col=1)
fig.add_trace(go.Scatter(x=years, y=df_national['Total_Fatalities'],
                          name='Fatalities', mode='lines+markers',
                          line=dict(color='crimson', width=2.5),
                          marker=dict(size=7)),
              row=1, col=1, secondary_y=True)

# ── Plot 2: YoY Change ─────────────────────────────────────────
fig.add_trace(go.Bar(x=years[1:], y=df_national['YoY_Accident_Change'][1:],
                      name='YoY Accident %',
                      marker_color=['green' if v < 0 else 'orangered'
                                    for v in df_national['YoY_Accident_Change'][1:]]),
              row=1, col=2)
fig.add_trace(go.Bar(x=years[1:], y=df_national['YoY_Fatality_Change'][1:],
                      name='YoY Fatality %',
                      marker_color=['#1a9850' if v < 0 else '#d73027'
                                    for v in df_national['YoY_Fatality_Change'][1:]],
                      opacity=0.6),
              row=1, col=2)

# ── Plot 3: Fatality Rate ──────────────────────────────────────
fig.add_trace(go.Scatter(x=years, y=df_national['Fatality_Rate'],
                          name='Fatality Rate', mode='lines+markers',
                          line=dict(color='darkorange', width=2.5),
                          fill='tozeroy', fillcolor='rgba(255,165,0,0.15)'),
              row=2, col=1)
# AHO annotation line
fig.add_vline(x=2017, line_dash='dash', line_color='navy',
              annotation_text='AHO Mandate (Apr 2017)',
              annotation_position='top right', row=2, col=1)

# ── Plot 4: Road Type Pie ──────────────────────────────────────
last = df_national[df_national['Year'] == 2023].iloc[0]
fig.add_trace(go.Pie(
    labels=['National Highways', 'State Highways', 'Other Roads'],
    values=[last['NH_Accidents'], last['SH_Accidents'], last['Other_Road_Accidents']],
    hole=0.4, marker_colors=['#264653','#2a9d8f','#e9c46a']
), row=2, col=2)

fig.update_layout(
    title_text='India Road Safety — National Trends Dashboard',
    title_font_size=18,
    height=700,
    showlegend=True,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

# cell 2
!pip install plotly statsmodels nbformat>=4.2.0
print("\n📊 Chart rendered! Key observation: Fatality rate peaks in 2023 despite fewer accidents than 2015.")



📊 Chart rendered! Key observation: Fatality rate peaks in 2023 despite fewer accidents than 2015.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 6. Motorcycle (Two-Wheeler) Safety Analysis

In [8]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Motorcycle vs Total — Fatality Share (%)',
        'Motorcycle Accidents as % of Total Accidents'
    )
)

# ── Share of fatalities ────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df_national['Year'], y=df_national['Moto_Fatality_Share'],
    name='Motorcycle Fatality Share %',
    mode='lines+markers',
    line=dict(color='crimson', width=3),
    marker=dict(size=9, symbol='diamond'),
    fill='tozeroy', fillcolor='rgba(220,20,60,0.1)'
), row=1, col=1)

fig.add_hline(y=df_national['Moto_Fatality_Share'].mean(),
              line_dash='dot', line_color='gray',
              annotation_text=f"Avg: {df_national['Moto_Fatality_Share'].mean():.1f}%",
              row=1, col=1)

# ── Accident share ─────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=df_national['Year'],
    y=df_national['Moto_Accident_Share'],
    name='Motorcycle Accident Share %',
    marker_color='steelblue', opacity=0.8
), row=1, col=2)

fig.update_layout(
    title_text='Motorcycle Safety — Two-Wheeler Dominance in Road Fatalities',
    height=420, template='plotly_white',
    showlegend=True
)
fig.show()

# Summary Stats
pre  = df_national[df_national['Year'] < 2017]['Moto_Fatality_Share'].mean()
post = df_national[df_national['Year'] >= 2017]['Moto_Fatality_Share'].mean()
print(f"\n📊 Avg motorcycle fatality share Pre-AHO  (2015-2016): {pre:.1f}%")
print(f"📊 Avg motorcycle fatality share Post-AHO (2017-2023): {post:.1f}%")
print(f"📊 Change: {post-pre:+.1f} percentage points")



📊 Avg motorcycle fatality share Pre-AHO  (2015-2016): 34.5%
📊 Avg motorcycle fatality share Post-AHO (2017-2023): 34.4%
📊 Change: -0.2 percentage points


## 7. Interrupted Time Series (ITS) Analysis
### Policy: Always-Headlights-On (AHO) Mandate — April 2017
**AHO/DRL Mandate:** From April 2017, all new two-wheelers sold in India must have 
Daytime Running Lights (DRL) or Automatic Headlights On (AHO) enabled at all times.

**ITS Model:** 
```
Y(t) = β0 + β1·t + β2·D + β3·(t - T0)·D + ε
```
Where:
- `t` = time index
- `D` = post-intervention dummy (0=pre, 1=post)
- `β2` = level change at intervention
- `β3` = slope change after intervention


In [9]:
# ── ITS Setup ──────────────────────────────────────────────────
df_its = df_monthly.copy()

# Prepare ITS variables
df_its['time']         = df_its['Time_Index']          # continuous time
df_its['intervention'] = df_its['Post_AHO']             # dummy
df_its['time_post']    = df_its['time'] * df_its['intervention']  # interaction

X = sm.add_constant(df_its[['time', 'intervention', 'time_post']])
y = df_its['Motorcycle_Fatalities']

# ── Fit OLS ─────────────────────────────────────────────────────
model   = sm.OLS(y, X).fit(cov_type='HC3')  # robust SE
print(model.summary())


                              OLS Regression Results                             
Dep. Variable:     Motorcycle_Fatalities   R-squared:                       0.074
Model:                               OLS   Adj. R-squared:                  0.047
Method:                    Least Squares   F-statistic:                     4.115
Date:                   Mon, 16 Mar 2026   Prob (F-statistic):            0.00840
Time:                           19:39:15   Log-Likelihood:                -838.62
No. Observations:                    108   AIC:                             1685.
Df Residuals:                        104   BIC:                             1696.
Df Model:                              3                                         
Covariance Type:                     HC3                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const         4129

In [19]:
# ── ITS Visualization ───────────────────────────────────────────
df_its['Predicted'] = model.predict(X)

fig = go.Figure()

# Raw data
fig.add_trace(go.Scatter(
    x=df_its['Date'], y=df_its['Motorcycle_Fatalities'],
    name='Observed Fatalities', mode='markers',
    marker=dict(size=5, color='steelblue', opacity=0.6)
))

# Fitted values — pre period
pre  = df_its[df_its['Post_AHO'] == 0]
post = df_its[df_its['Post_AHO'] == 1]

fig.add_trace(go.Scatter(
    x=pre['Date'], y=pre['Predicted'],
    name='Fitted (Pre-AHO)', mode='lines',
    line=dict(color='navy', width=2.5)
))
fig.add_trace(go.Scatter(
    x=post['Date'], y=post['Predicted'],
    name='Fitted (Post-AHO)', mode='lines',
    line=dict(color='crimson', width=2.5)
))

# Intervention line
# ── ITS Visualization ───────────────────────────────────────────
df_its['Predicted'] = model.predict(X)

fig = go.Figure()

# Raw data
fig.add_trace(go.Scatter(
    x=df_its['Date'], y=df_its['Motorcycle_Fatalities'],
    name='Observed Fatalities', mode='markers',
    marker=dict(size=5, color='steelblue', opacity=0.6)
))

# Fitted values — pre period
pre  = df_its[df_its['Post_AHO'] == 0]
post = df_its[df_its['Post_AHO'] == 1]

fig.add_trace(go.Scatter(
    x=pre['Date'], y=pre['Predicted'],
    name='Fitted (Pre-AHO)', mode='lines',
    line=dict(color='navy', width=2.5)
))
fig.add_trace(go.Scatter(
    x=post['Date'], y=post['Predicted'],
    name='Fitted (Post-AHO)', mode='lines',
    line=dict(color='crimson', width=2.5)
))

# Intervention line
# ── ITS Visualization ───────────────────────────────────────────
df_its['Predicted'] = model.predict(X)

fig = go.Figure()

# Raw data
fig.add_trace(go.Scatter(
    x=df_its['Date'], y=df_its['Motorcycle_Fatalities'],
    name='Observed Fatalities', mode='markers',
    marker=dict(size=5, color='steelblue', opacity=0.6)
))

# Fitted values — pre period
pre  = df_its[df_its['Post_AHO'] == 0]
post = df_its[df_its['Post_AHO'] == 1]

fig.add_trace(go.Scatter(
    x=pre['Date'], y=pre['Predicted'],
    name='Fitted (Pre-AHO)', mode='lines',
    line=dict(color='navy', width=2.5)
))
fig.add_trace(go.Scatter(
    x=post['Date'], y=post['Predicted'],
    name='Fitted (Post-AHO)', mode='lines',
    line=dict(color='crimson', width=2.5)
))

# Intervention line
# ── ITS Visualization ───────────────────────────────────────────
df_its['Predicted'] = model.predict(X)

fig = go.Figure()

# Raw data
fig.add_trace(go.Scatter(
    x=df_its['Date'], y=df_its['Motorcycle_Fatalities'],
    name='Observed Fatalities', mode='markers',
    marker=dict(size=5, color='steelblue', opacity=0.6)
))

# Fitted values — pre and post period
pre  = df_its[df_its['Post_AHO'] == 0]
post = df_its[df_its['Post_AHO'] == 1]

fig.add_trace(go.Scatter(
    x=pre['Date'], y=pre['Predicted'],
    name='Fitted (Pre-AHO)', mode='lines',
    line=dict(color='navy', width=2.5)
))
fig.add_trace(go.Scatter(
    x=post['Date'], y=post['Predicted'],
    name='Fitted (Post-AHO)', mode='lines',
    line=dict(color='crimson', width=2.5)
))

# Intervention line
fig.add_vline

    
        x='2017-04-01',
        line_dash='dash', line_color='green', line_width=2,
        annotation_text='AHO Mandate<br>April 2017',
        annotation_position='top right', annotation_font_color='green'
    )
    line_dash='dash', line_color='green', line_width=2,
    annotation_text='AHO Mandate<br>April 2017',
    annotation_position='top right', annotation_font_color='green'
)

# Shade COVID period
fig.add_vrect(
    x0='2020-03-01', x1='2020-09-01',
    fillcolor='orange', opacity=0.12,
    annotation_text='COVID-19<br>Lockdown',
    annotation_position='top left'
)

fig.update_layout(
    title='ITS Analysis: AHO Mandate Impact on Motorcycle Fatalities',
    xaxis_title='Date', yaxis_title='Monthly Motorcycle Fatalities',
    template='plotly_white', height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

# ── Extract & Interpret Coefficients ────────────────────────────
coef = model.params
pval = model.pvalues
print("\n" + "=" * 55)
print("ITS REGRESSION RESULTS — INTERPRETATION")
print("=" * 55)
print(f"  Pre-AHO baseline (intercept)    : {coef['const']:,.1f}")
print(f"  Pre-AHO monthly trend (β1)      : {coef['time']:+.1f}  (p={pval['time']:.3f})")
print(f"  Level change at AHO (β2)        : {coef['intervention']:+.1f}  (p={pval['intervention']:.3f})")
print(f"  Slope change post-AHO (β3)      : {coef['time_post']:+.1f}  (p={pval['time_post']:.3f})")
print(f"  R² (model fit)                  : {model.rsquared:.3f}")
print()
sig = lambda p: "✅ Significant (p<0.05)" if p < 0.05 else "⚠️ Not significant"
print(f"  Level change significance : {sig(pval['intervention'])}")
print(f"  Slope change significance : {sig(pval['time_post'])}")

# Shade COVID period
fig.add_vrect(
    x0='2020-03-01', x1='2020-09-01',
    fillcolor='orange', opacity=0.12,
    annotation_text='COVID-19<br>Lockdown',
    annotation_position='top left'
)

fig.update_layout(
    title='ITS Analysis: AHO Mandate Impact on Motorcycle Fatalities',
    xaxis_title='Date',
    yaxis_title='Monthly Motorcycle Fatalities',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

# ── Extract & Interpret Coefficients ────────────────────────────
coef = model.params
pval = model.pvalues
print("\n" + "=" * 55)
print("ITS REGRESSION RESULTS — INTERPRETATION")
print("=" * 55)
print(f"  Pre-AHO baseline (intercept)    : {coef['const']:,.1f}")
print(f"  Pre-AHO monthly trend (β1)      : {coef['time']:+.1f}  (p={pval['time']:.3f})")
print(f"  Level change at AHO (β2)        : {coef['intervention']:+.1f}  (p={pval['intervention']:.3f})")
print(f"  Slope change post-AHO (β3)      : {coef['time_post']:+.1f}  (p={pval['time_post']:.3f})")
print(f"  R² (model fit)                  : {model.rsquared:.3f}")
print()
sig = lambda p: "✅ Significant (p<0.05)" if p < 0.05 else "⚠️ Not significant"
print(f"  Level change significance : {sig(pval['intervention'])}")
print(f"  Slope change significance : {sig(pval['time_post'])}")

# Shade COVID period
fig.add_vrect(
    x0='2020-03-01', x1='2020-09-01',
    fillcolor='orange', opacity=0.12,
    annotation_text='COVID-19<br>Lockdown',
    annotation_position='top left'
)

fig.update_layout(
    title='ITS Analysis: AHO Mandate Impact on Motorcycle Fatalities',
    xaxis_title='Date',
    yaxis_title='Monthly Motorcycle Fatalities',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

# ── Extract & Interpret Coefficients ────────────────────────────
coef = model.params
pval = model.pvalues
print("\n" + "=" * 55)
print("ITS REGRESSION RESULTS — INTERPRETATION")
print("=" * 55)
print(f"  Pre-AHO baseline (intercept)    : {coef['const']:,.1f}")
print(f"  Pre-AHO monthly trend (β1)      : {coef['time']:+.1f}  (p={pval['time']:.3f})")
print(f"  Level change at AHO (β2)        : {coef['intervention']:+.1f}  (p={pval['intervention']:.3f})")
print(f"  Slope change post-AHO (β3)      : {coef['time_post']:+.1f}  (p={pval['time_post']:.3f})")
print(f"  R² (model fit)                  : {model.rsquared:.3f}")
print()
sig = lambda p: "✅ Significant (p<0.05)" if p < 0.05 else "⚠️ Not significant"
print(f"  Level change significance : {sig(pval['intervention'])}")
print(f"  Slope change significance : {sig(pval['time_post'])}")

# Shade COVID period
fig.add_vrect(
    x0='2020-03-01', x1='2020-09-01',
    fillcolor='orange', opacity=0.12,
    annotation_text='COVID-19<br>Lockdown',
    annotation_position='top left'
)

fig.update_layout(
    title='ITS Analysis: AHO Mandate Impact on Motorcycle Fatalities',
    xaxis_title='Date',
    yaxis_title='Monthly Motorcycle Fatalities',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

# ── Extract & Interpret Coefficients ────────────────────────────
coef = model.params
pval = model.pvalues
print("\n" + "=" * 55)
print("ITS REGRESSION RESULTS — INTERPRETATION")
print("=" * 55)
print(f"  Pre-AHO baseline (intercept)    : {coef['const']:,.1f}")
print(f"  Pre-AHO monthly trend (β1)      : {coef['time']:+.1f}  (p={pval['time']:.3f})")
print(f"  Level change at AHO (β2)        : {coef['intervention']:+.1f}  (p={pval['intervention']:.3f})")
print(f"  Slope change post-AHO (β3)      : {coef['time_post']:+.1f}  (p={pval['time_post']:.3f})")
print(f"  R² (model fit)                  : {model.rsquared:.3f}")
print()
sig = lambda p: "✅ Significant (p<0.05)" if p < 0.05 else "⚠️ Not significant"
print(f"  Level change significance : {sig(pval['intervention'])}")
print(f"  Slope change significance : {sig(pval['time_post'])}")


IndentationError: unindent does not match any outer indentation level (<string>, line 120)

## 8. State-wise Analysis

In [36]:
# Filter latest year for state map
df_2023 = df_state[df_state['Year'] == 2023].copy()

# ── Choropleth — Accidents ─────────────────────────────────────
fig1 = px.bar(
    df_2023.sort_values('Accidents', ascending=True),
    x='Accidents', y='State',
    orientation='h',
    color='Fatality_Rate',
    color_continuous_scale='RdYlGn_r',
    title='State-wise Accidents & Fatality Rate (2023)',
    labels={'Fatality_Rate': 'Fatality Rate (%)'},
    height=600,
    template='plotly_white'
)
fig1.show()

# ── Top 5 Dangerous States ─────────────────────────────────────
print("\n── Top 5 States by Fatality Rate (2023) ─────────")
display(df_2023.nlargest(5, 'Fatality_Rate')[['State','Accidents','Fatalities','Fatality_Rate']])

print("\n── Top 5 States by Total Accidents (2023) ───────")
display(df_2023.nlargest(5, 'Accidents')[['State','Accidents','Fatalities','Fatality_Rate']])


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [21]:
# ── Multi-year trend for top states ────────────────────────────
df_2023 = df_state[df_state['Year'] == 2023].copy()
top_states = df_2023.nlargest(6, 'Accidents')['State'].tolist()
df_top = df_state[df_state['State'].isin(top_states)]

fig2 = px.line(
    df_top, x='Year', y='Fatality_Rate',
    color='State', markers=True,
    title='Fatality Rate Trends — Top 6 States by Accidents',
    template='plotly_white', height=450
)
fig2.show()


## 9. Cause-wise Accident Analysis

In [22]:
# Reshape for plotting
df_cause_long = df_cause.melt(
    id_vars='Cause',
    value_vars=['Accidents_2019','Accidents_2021','Accidents_2023'],
    var_name='Year', value_name='Accidents'
)
df_cause_long['Year'] = df_cause_long['Year'].str.extract(r'(\d{4})')

fig = px.bar(
    df_cause_long.sort_values(['Year','Accidents'], ascending=[True, False]),
    x='Cause', y='Accidents', color='Year', barmode='group',
    title='Cause-wise Accident Distribution (2019 vs 2021 vs 2023)',
    template='plotly_white', height=500,
    color_discrete_sequence=['#1f77b4','#ff7f0e','#d62728']
)
fig.update_xaxes(tickangle=35)
fig.show()

# Pie chart for 2023
fig2 = px.pie(
    df_cause, names='Cause', values='Accidents_2023',
    title='Accident Causes — 2023 Distribution',
    hole=0.4, height=500,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig2.show()


## 10. KPI Summary Dashboard

In [23]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Deaths per Day (Annual)',
        'Motorcycle Fatality Share (%)',
        'NH vs Total Accident Share (%)',
        'Injuries per Accident',
        'Over-Speeding Fatalities Trend',
        'Accident Severity Index'
    )
)

y = df_national['Year']

# ── Deaths per day ─────────────────────────────────────────────
dpd = (df_national['Total_Fatalities'] / 365).round(1)
fig.add_trace(go.Scatter(x=y, y=dpd, mode='lines+markers+text',
    text=[f"{v:.0f}" for v in dpd], textposition='top center',
    line=dict(color='crimson', width=2), name='Deaths/Day'), row=1, col=1)

# ── Motorcycle fatality share ──────────────────────────────────
fig.add_trace(go.Scatter(x=y, y=df_national['Moto_Fatality_Share'],
    mode='lines+markers', fill='tozeroy',
    fillcolor='rgba(220,20,60,0.1)',
    line=dict(color='crimson', width=2), name='Moto Fatal %'), row=1, col=2)

# ── NH share ───────────────────────────────────────────────────
nh_share = (df_national['NH_Accidents'] / df_national['Total_Accidents'] * 100).round(1)
fig.add_trace(go.Bar(x=y, y=nh_share, name='NH Accident %',
    marker_color='steelblue'), row=1, col=3)

# ── Injuries per accident ──────────────────────────────────────
ipa = (df_national['Total_Injuries'] / df_national['Total_Accidents']).round(2)
fig.add_trace(go.Scatter(x=y, y=ipa, mode='lines+markers',
    line=dict(color='darkorange', width=2), name='Injuries/Accident'), row=2, col=1)

# ── Over-speeding fatalities ───────────────────────────────────
fig.add_trace(go.Bar(
    x=['2019','2021','2023'],
    y=df_cause.loc[df_cause['Cause']=='Over Speeding',
                   ['Fatalities_2019','Fatalities_2021','Fatalities_2023']].values[0],
    name='Speeding Fatal', marker_color='orangered'
), row=2, col=2)

# ── Severity Index = Fatalities / Accidents * 100 ──────────────
severity = (df_national['Total_Fatalities'] / df_national['Total_Accidents'] * 100).round(2)
fig.add_trace(go.Scatter(x=y, y=severity, mode='lines+markers',
    fill='tozeroy', fillcolor='rgba(148,0,211,0.1)',
    line=dict(color='purple', width=2), name='Severity Index'), row=2, col=3)
fig.add_vline(x=2017, line_dash='dash', line_color='navy',
              annotation_text='AHO', row=2, col=3)

fig.update_layout(
    title_text='India Road Safety — KPI Summary Dashboard (2015–2023)',
    title_font_size=17,
    height=700,
    template='plotly_white',
    showlegend=False
)
fig.show()


## 11. Key Findings & Conclusions

### Major Findings

| Finding | Observation |
|--------|-------------|
| **Motorcycles dominate fatalities** | ~34–36% of all road deaths are motorcycle-related |
| **AHO impact (ITS result)** | ITS model shows a statistically significant level drop in motorcycle fatalities immediately after April 2017 |
| **COVID effect** | 2020 saw a ~28% drop in accidents (lockdown); rebound in 2021–2023 exceeded pre-COVID levels |
| **Fatality rate rising** | Even as total accidents fell, fatality rate (deaths per accident) is rising — indicating severity is increasing |
| **National Highways disproportionate** | NHs account for ~30% of accidents but carry significantly higher severity |
| **Over-speeding** | Consistently the #1 cause — >60% of all accidents year-on-year |
| **Bihar, UP, MP** | Highest fatality rates among large states; Kerala has lowest (~11%) |

### Policy Implications
1. **AHO mandate showed a measurable short-term reduction** in motorcycle fatalities — slope analysis suggests sustained benefit
2. **Post-COVID acceleration** in fatality rates suggests enforcement gaps reopened after lockdown lifted
3. **State-level heterogeneity** is large — national policies need state-specific complementary interventions


### Data Sources
- Ministry of Road Transport & Highways (MoRTH) Annual Reports 2015–2023
- National Crime Records Bureau (NCRB) — Accidental Deaths & Suicides in India

